# 🎨 Paint by Number Image Converter

**Author:** AI Expert  
**Description:** This notebook converts any image into a **Paint-by-Number** format using open-source computer vision techniques.  
Each region is assigned a number and a suggested color, making it ready to print and paint!

---

## 📦 Pipeline Overview
1. **Install & Import** dependencies
2. **Color Quantization** — reduce image to N dominant colors using K-Means
3. **Region Segmentation** — identify contiguous color regions
4. **Contour Drawing** — draw black outlines around each region
5. **Number Labeling** — place a number inside each region
6. **Color Legend** — generate a printable color palette guide
7. **Groq AI Integration** — optionally use Groq LLM to suggest artistic color names
8. **Gradio App** — interactive professional web UI

---

## 📦 Section 1: Install Dependencies

In [ ]:
# ============================================================
# SECTION 1: INSTALL DEPENDENCIES
# ============================================================
# Installs all required Python packages for image processing,
# UI deployment, and Groq API integration.
# ============================================================

!pip install opencv-python-headless numpy Pillow scikit-learn scipy matplotlib groq gradio requests -q
print("✅ All dependencies installed successfully!")

## 📚 Section 2: Import Libraries

In [ ]:
# ============================================================
# SECTION 2: IMPORT LIBRARIES
# ============================================================
# Imports all necessary libraries used throughout the pipeline:
# - cv2       : OpenCV for image processing and contour detection
# - numpy     : Numerical operations and array manipulation
# - PIL       : Pillow for image loading, saving, and conversion
# - sklearn   : K-Means clustering for color quantization
# - scipy     : Connected component labeling for region detection
# - matplotlib: Visualizing results and color legends
# - groq      : Groq API client for LLM-based color naming
# - gradio    : Web UI deployment framework
# ============================================================

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from sklearn.cluster import KMeans
from scipy import ndimage
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_hex
import os
import io
import base64
import tempfile
from pathlib import Path
import gradio as gr
from groq import Groq
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## 🔧 Section 3: Core Image Processing Functions

In [ ]:
# ============================================================
# SECTION 3: CORE IMAGE PROCESSING FUNCTIONS
# ============================================================


def load_and_preprocess_image(image_input, max_size=800):
    """
    Load and preprocess an input image for paint-by-number conversion.

    This function accepts either a file path (str) or a PIL Image object,
    converts it to RGB format, and optionally resizes it so that its
    longest dimension does not exceed `max_size` pixels. Resizing helps
    reduce computation time for large images without sacrificing quality.

    Parameters:
    -----------
    image_input : str or PIL.Image.Image
        Path to the image file, or a PIL Image object already in memory.
    max_size : int, optional (default=800)
        Maximum allowed dimension (width or height) in pixels.
        If the image exceeds this, it is downscaled proportionally.

    Returns:
    --------
    PIL.Image.Image
        A preprocessed RGB PIL Image ready for quantization.
    """
    if isinstance(image_input, str):
        img = Image.open(image_input).convert('RGB')
    elif isinstance(image_input, np.ndarray):
        img = Image.fromarray(image_input).convert('RGB')
    else:
        img = image_input.convert('RGB')

    # Resize proportionally if image is too large
    w, h = img.size
    if max(w, h) > max_size:
        scale = max_size / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        img = img.resize((new_w, new_h), Image.LANCZOS)
    return img


def quantize_colors(img, n_colors=12):
    """
    Reduce an image to a fixed number of dominant colors using K-Means clustering.

    Each pixel in the image is treated as a point in 3D RGB color space.
    K-Means groups all pixels into `n_colors` clusters, then replaces
    every pixel with its cluster centroid color. This creates flat color
    regions that are ideal for paint-by-number outlines.

    Parameters:
    -----------
    img : PIL.Image.Image
        The source RGB image to quantize.
    n_colors : int, optional (default=12)
        The number of distinct colors (paint buckets) to produce.
        Fewer colors = simpler painting; more colors = more detail.

    Returns:
    --------
    quantized_img : numpy.ndarray  (H x W x 3, uint8)
        The color-quantized image where every pixel is one of the cluster colors.
    labels_2d : numpy.ndarray  (H x W, int)
        A 2D array of cluster indices (0 to n_colors-1) for each pixel.
    palette : numpy.ndarray  (n_colors x 3, float)
        The RGB centroid colors (0–255 range) for each cluster.
    """
    img_array = np.array(img, dtype=np.float32)
    h, w, c = img_array.shape
    pixels = img_array.reshape(-1, 3)

    # Apply K-Means clustering in RGB space
    kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10)
    kmeans.fit(pixels)

    labels = kmeans.labels_
    palette = kmeans.cluster_centers_  # shape: (n_colors, 3)

    # Reconstruct the quantized image
    quantized_pixels = palette[labels].astype(np.uint8)
    quantized_img = quantized_pixels.reshape(h, w, 3)
    labels_2d = labels.reshape(h, w)

    return quantized_img, labels_2d, palette


def apply_smoothing(labels_2d, iterations=2):
    """
    Smooth the segmentation label map to reduce noisy, fragmented regions.

    Small isolated pixels belonging to a label can produce messy, unnatural
    regions. This function applies morphological opening (erosion + dilation)
    to each label plane, effectively removing tiny specks and smoothing
    jagged region boundaries for a cleaner paint-by-number result.

    Parameters:
    -----------
    labels_2d : numpy.ndarray  (H x W, int)
        2D array of cluster label indices per pixel.
    iterations : int, optional (default=2)
        Number of morphological open/close passes to apply.
        Higher values smooth more aggressively.

    Returns:
    --------
    numpy.ndarray  (H x W, int)
        Smoothed label map with reduced noise and cleaner boundaries.
    """
    n_colors = labels_2d.max() + 1
    smoothed = labels_2d.copy()
    kernel = np.ones((5, 5), np.uint8)

    for label_id in range(n_colors):
        mask = (labels_2d == label_id).astype(np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=iterations)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)
        smoothed[mask == 1] = label_id

    return smoothed


print("✅ Core image processing functions defined!")

## ✏️ Section 4: Contour & Outline Drawing Functions

In [ ]:
# ============================================================
# SECTION 4: CONTOUR & OUTLINE DRAWING FUNCTIONS
# ============================================================


def draw_outlines(labels_2d):
    """
    Generate a black-and-white outline image by detecting region boundaries.

    This function identifies every pixel where a label changes to an adjacent
    label (i.e., a region boundary) using horizontal and vertical gradient
    detection. The result is a binary mask where boundary pixels are black
    (0) and interior pixels are white (255) — the classic paint-by-number
    coloring page look.

    Parameters:
    -----------
    labels_2d : numpy.ndarray  (H x W, int)
        2D array of cluster label indices per pixel.

    Returns:
    --------
    outline_img : numpy.ndarray  (H x W x 3, uint8)
        RGB image with black region boundaries on a white background.
    """
    h, w = labels_2d.shape
    outline_mask = np.zeros((h, w), dtype=np.uint8)

    # Detect horizontal and vertical boundaries
    diff_h = np.diff(labels_2d, axis=0) != 0
    diff_v = np.diff(labels_2d, axis=1) != 0

    outline_mask[:-1, :] |= diff_h
    outline_mask[1:, :]  |= diff_h
    outline_mask[:, :-1] |= diff_v
    outline_mask[:, 1:]  |= diff_v

    # Convert to white background with black outlines
    outline_img = np.ones((h, w, 3), dtype=np.uint8) * 255
    outline_img[outline_mask == 1] = [0, 0, 0]

    return outline_img


def find_region_centers(labels_2d, min_area=200):
    """
    Locate the visual center of each connected region for number placement.

    For each color label, the function extracts all connected components
    (i.e., contiguous blobs of the same color). For regions large enough
    (area >= min_area), it computes the centroid using image moments.
    These centroids are where the paint numbers will be drawn.

    Parameters:
    -----------
    labels_2d : numpy.ndarray  (H x W, int)
        2D array of cluster label indices per pixel.
    min_area : int, optional (default=200)
        Minimum pixel area a region must have to receive a number label.
        Filters out tiny regions that would be too small to paint.

    Returns:
    --------
    list of tuples : [(label_id, cx, cy, area), ...]
        Each tuple contains the label index, centroid x/y coordinates,
        and the pixel area of the region.
    """
    n_colors = labels_2d.max() + 1
    region_centers = []

    for label_id in range(n_colors):
        mask = (labels_2d == label_id).astype(np.uint8)
        # Find connected components for this label
        num_components, component_map = cv2.connectedComponents(mask)

        for comp_id in range(1, num_components):
            comp_mask = (component_map == comp_id).astype(np.uint8)
            area = comp_mask.sum()
            if area < min_area:
                continue
            # Compute centroid using image moments
            M = cv2.moments(comp_mask)
            if M['m00'] == 0:
                continue
            cx = int(M['m10'] / M['m00'])
            cy = int(M['m01'] / M['m00'])
            region_centers.append((label_id, cx, cy, area))

    return region_centers


def draw_numbers_on_outline(outline_img, region_centers, font_scale=0.4, thickness=1):
    """
    Overlay color-index numbers onto the outline image at region centers.

    Iterates over all detected region centers and draws the corresponding
    color number (1-indexed) using OpenCV's putText. Numbers are drawn in
    dark gray so they are visible on the white background but don't
    dominate the image. The number size adapts based on font_scale.

    Parameters:
    -----------
    outline_img : numpy.ndarray  (H x W x 3, uint8)
        The white-background outline image to draw numbers onto.
    region_centers : list of tuples
        Output of find_region_centers(): [(label_id, cx, cy, area), ...]
    font_scale : float, optional (default=0.4)
        OpenCV font scale factor controlling number size.
    thickness : int, optional (default=1)
        Line thickness for drawing the number text.

    Returns:
    --------
    numpy.ndarray  (H x W x 3, uint8)
        The outline image with numbers drawn at each valid region center.
    """
    numbered_img = outline_img.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX

    for (label_id, cx, cy, area) in region_centers:
        number_text = str(label_id + 1)  # 1-indexed for user display
        # Scale font size with region area
        adaptive_scale = min(font_scale + area / 50000, 0.9)
        text_size = cv2.getTextSize(number_text, font, adaptive_scale, thickness)[0]
        text_x = cx - text_size[0] // 2
        text_y = cy + text_size[1] // 2
        # Draw with a subtle white halo for readability
        cv2.putText(numbered_img, number_text, (text_x, text_y),
                    font, adaptive_scale, (255, 255, 255), thickness + 2)
        cv2.putText(numbered_img, number_text, (text_x, text_y),
                    font, adaptive_scale, (50, 50, 50), thickness)

    return numbered_img


print("✅ Contour & outline drawing functions defined!")

## 🎨 Section 5: Color Legend & Palette Functions

In [ ]:
# ============================================================
# SECTION 5: COLOR LEGEND & PALETTE FUNCTIONS
# ============================================================


def rgb_to_hex(rgb):
    """
    Convert an RGB tuple or array to a CSS-style hex color string.

    Parameters:
    -----------
    rgb : array-like of 3 numbers (R, G, B) in range 0–255

    Returns:
    --------
    str : Hex color code like '#A3C2F0'
    """
    r, g, b = int(rgb[0]), int(rgb[1]), int(rgb[2])
    return f'#{r:02X}{g:02X}{b:02X}'


def get_color_name_from_rgb(rgb):
    """
    Generate a descriptive artistic color name from an RGB value using heuristics.

    This function uses HSV (Hue-Saturation-Value) analysis to categorize
    each color into a human-friendly paint name (e.g., 'Sky Blue',
    'Deep Forest Green', 'Warm Ivory'). It handles edge cases like
    near-white, near-black, and desaturated grays separately.

    Parameters:
    -----------
    rgb : array-like  (R, G, B in 0–255)

    Returns:
    --------
    str : A descriptive color name suitable for display in a paint legend.
    """
    r, g, b = int(rgb[0]), int(rgb[1]), int(rgb[2])

    # Convert to HSV for better perceptual classification
    pixel = np.uint8([[[r, g, b]]])
    hsv = cv2.cvtColor(pixel, cv2.COLOR_RGB2HSV)[0][0]
    h, s, v = int(hsv[0]), int(hsv[1]), int(hsv[2])

    # Handle near-black
    if v < 40:
        return "Deep Shadow Black"
    # Handle near-white
    if v > 220 and s < 30:
        return "Warm Canvas White"
    # Handle grays
    if s < 40:
        if v < 100: return "Charcoal Gray"
        if v < 170: return "Stone Gray"
        return "Light Silver"

    # Brightness modifiers
    if v < 80:   brightness = "Dark "
    elif v > 200: brightness = "Light "
    else:         brightness = ""

    # Saturation modifiers
    if s < 80:    saturation = "Muted "
    elif s > 200: saturation = "Vivid "
    else:         saturation = ""

    # Hue-based name mapping (OpenCV HSV hue range: 0-179)
    if h < 10 or h >= 170:  base = "Red"
    elif h < 20:            base = "Orange"
    elif h < 30:            base = "Amber"
    elif h < 40:            base = "Yellow"
    elif h < 55:            base = "Lime Green"
    elif h < 75:            base = "Green"
    elif h < 90:            base = "Forest Green"
    elif h < 105:           base = "Teal"
    elif h < 115:           base = "Cyan"
    elif h < 130:           base = "Sky Blue"
    elif h < 145:           base = "Cobalt Blue"
    elif h < 155:           base = "Indigo"
    elif h < 165:           base = "Violet"
    else:                   base = "Magenta"

    return f"{brightness}{saturation}{base}"


def get_color_names_from_groq(palette, api_key):
    """
    Use the Groq LLM API to generate creative artistic paint color names.

    Sends all palette RGB values to Groq's fast LLM inference in a single
    request. The model generates evocative, artist-quality names for each
    color (e.g., 'Twilight Lavender', 'Burnt Sienna Earth'). Falls back
    gracefully to the local heuristic naming if the API call fails.

    Parameters:
    -----------
    palette : numpy.ndarray  (N x 3)
        Array of RGB color values from K-Means centroids.
    api_key : str
        A valid Groq API key for authentication.

    Returns:
    --------
    list of str
        One color name per palette entry. Falls back to heuristic names
        if the API call fails or the key is invalid.
    """
    if not api_key or api_key.strip() == "":
        return [get_color_name_from_rgb(c) for c in palette]

    try:
        client = Groq(api_key=api_key.strip())
        color_list = ", ".join(
            [f"Color {i+1}: RGB({int(c[0])},{int(c[1])},{int(c[2])})" for i, c in enumerate(palette)]
        )
        prompt = (
            f"You are an expert fine art color consultant. I have {len(palette)} paint colors:\n{color_list}.\n\n"
            "For each color, provide ONE evocative, artistic paint color name (like those used by premium art supply brands). "
            "Reply with ONLY a JSON array of strings in order, no extra text. Example: [\"Tuscan Sunset\", \"Midnight Teal\"]"
        )
        response = client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
            temperature=0.7
        )
        import json, re
        raw = response.choices[0].message.content.strip()
        # Extract JSON array from response
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        if match:
            names = json.loads(match.group())
            if len(names) == len(palette):
                return names
    except Exception as e:
        print(f"⚠️ Groq API fallback to heuristics: {e}")

    return [get_color_name_from_rgb(c) for c in palette]


def create_color_legend(palette, color_names, cols=4):
    """
    Render a printable paint color legend as a PIL Image.

    Creates a visually clean legend grid showing each color's number,
    a color swatch, its hex code, and its descriptive name. The legend
    is designed to be printed alongside the numbered outline image so
    artists know exactly which paint to use for each number.

    Parameters:
    -----------
    palette : numpy.ndarray  (N x 3)
        Array of RGB color values.
    color_names : list of str
        Descriptive name for each palette color.
    cols : int, optional (default=4)
        Number of color swatches per row in the legend grid.

    Returns:
    --------
    PIL.Image.Image
        A rendered RGBA legend image ready to save or display.
    """
    n = len(palette)
    rows = (n + cols - 1) // cols

    swatch_size = 60
    cell_w, cell_h = 220, 80
    padding = 20
    header_h = 60

    total_w = cols * cell_w + 2 * padding
    total_h = rows * cell_h + header_h + 2 * padding

    legend_img = Image.new('RGB', (total_w, total_h), color=(248, 246, 240))
    draw = ImageDraw.Draw(legend_img)

    # Header
    draw.rectangle([0, 0, total_w, header_h], fill=(40, 40, 40))
    draw.text((total_w // 2, header_h // 2), "🎨 Paint Color Legend",
               fill=(255, 230, 100), anchor="mm",
               font=ImageFont.load_default())

    for i, (color, name) in enumerate(zip(palette, color_names)):
        row, col = divmod(i, cols)
        x = padding + col * cell_w
        y = header_h + padding + row * cell_h

        # Color swatch with border
        swatch_rect = [x, y + 10, x + swatch_size, y + 10 + swatch_size]
        draw.rectangle(swatch_rect, fill=tuple(int(v) for v in color))
        draw.rectangle(swatch_rect, outline=(0, 0, 0), width=2)

        # Number badge
        badge_rect = [x, y + 10, x + 22, y + 32]
        draw.rectangle(badge_rect, fill=(40, 40, 40))
        draw.text((x + 11, y + 21), str(i + 1), fill='white', anchor="mm",
                   font=ImageFont.load_default())

        # Color name and hex
        hex_code = rgb_to_hex(color)
        draw.text((x + swatch_size + 10, y + 18), name[:20],
                   fill=(40, 40, 40), font=ImageFont.load_default())
        draw.text((x + swatch_size + 10, y + 40), hex_code,
                   fill=(100, 100, 100), font=ImageFont.load_default())

    return legend_img


print("✅ Color legend & palette functions defined!")

## 🖼️ Section 6: Main Pipeline Orchestrator

In [ ]:
# ============================================================
# SECTION 6: MAIN PIPELINE ORCHESTRATOR
# ============================================================


def convert_to_paint_by_number(image_input, n_colors=12, groq_api_key="",
                                 min_region_area=200, smoothing=True):
    """
    Full end-to-end pipeline: convert an image into a paint-by-number output.

    Orchestrates all steps of the conversion:
    1. Preprocess the input image
    2. Quantize colors using K-Means
    3. Optionally smooth the segmentation
    4. Draw outlines between regions
    5. Find region centers for number placement
    6. Draw numbers onto the outline
    7. Generate color names (via Groq LLM or heuristics)
    8. Create a printable color legend
    9. Save and return output paths

    Parameters:
    -----------
    image_input : str or PIL.Image.Image or numpy.ndarray
        The source image (file path, PIL image, or NumPy array).
    n_colors : int, optional (default=12)
        Number of paint colors to use (2–24 recommended).
    groq_api_key : str, optional
        Groq API key for LLM-powered artistic color names.
        Pass empty string to use heuristic names.
    min_region_area : int, optional (default=200)
        Minimum pixel area for a region to receive a number label.
    smoothing : bool, optional (default=True)
        Whether to apply morphological smoothing to reduce noise.

    Returns:
    --------
    dict with keys:
        'outline_path'    : str — path to the numbered outline image
        'legend_path'     : str — path to the color legend image
        'quantized_path'  : str — path to the quantized reference image
        'color_names'     : list of str — color names for each number
        'palette'         : numpy.ndarray — RGB palette values
        'hex_codes'       : list of str — hex codes for each color
    """
    print("🔄 Step 1: Loading and preprocessing image...")
    img = load_and_preprocess_image(image_input)

    print(f"🔄 Step 2: Quantizing to {n_colors} colors...")
    quantized_img, labels_2d, palette = quantize_colors(img, n_colors)

    if smoothing:
        print("🔄 Step 3: Smoothing regions...")
        labels_2d = apply_smoothing(labels_2d)

    print("🔄 Step 4: Drawing outlines...")
    outline_img = draw_outlines(labels_2d)

    print("🔄 Step 5: Finding region centers...")
    region_centers = find_region_centers(labels_2d, min_area=min_region_area)

    print("🔄 Step 6: Numbering regions...")
    numbered_img = draw_numbers_on_outline(outline_img, region_centers)

    print("🔄 Step 7: Generating color names...")
    color_names = get_color_names_from_groq(palette, groq_api_key)
    hex_codes = [rgb_to_hex(c) for c in palette]

    print("🔄 Step 8: Creating color legend...")
    legend_img = create_color_legend(palette, color_names)

    print("🔄 Step 9: Saving outputs...")
    out_dir = tempfile.mkdtemp()
    outline_path   = os.path.join(out_dir, "paint_by_number_outline.png")
    legend_path    = os.path.join(out_dir, "paint_by_number_legend.png")
    quantized_path = os.path.join(out_dir, "paint_by_number_reference.png")

    Image.fromarray(numbered_img).save(outline_path)
    legend_img.save(legend_path)
    Image.fromarray(quantized_img).save(quantized_path)

    print(f"✅ Conversion complete! {len(region_centers)} regions numbered.")

    return {
        'outline_path'   : outline_path,
        'legend_path'    : legend_path,
        'quantized_path' : quantized_path,
        'color_names'    : color_names,
        'palette'        : palette,
        'hex_codes'      : hex_codes,
    }


print("✅ Main pipeline orchestrator defined!")

## 🧪 Section 7: Test the Pipeline (Quick Demo)

In [ ]:
# ============================================================
# SECTION 7: QUICK DEMO — TEST WITH A SAMPLE IMAGE
# ============================================================
# Creates a synthetic gradient test image and runs the full
# pipeline to verify everything works before using real images.
# ============================================================

def create_test_image(size=300):
    """
    Generate a colorful synthetic test image with gradient blobs.

    Creates a simple procedural image with color blobs arranged in
    a grid pattern. Used to verify the pipeline runs correctly
    without needing an external image file.

    Parameters:
    -----------
    size : int  (default=300)
        Width and height of the generated square test image.

    Returns:
    --------
    PIL.Image.Image : Synthetic RGB test image.
    """
    img_array = np.zeros((size, size, 3), dtype=np.uint8)
    colors_test = [
        (220, 80, 80),   # Red
        (80, 160, 220),  # Blue
        (80, 200, 100),  # Green
        (240, 200, 60),  # Yellow
        (200, 100, 200), # Purple
        (255, 160, 40),  # Orange
    ]
    for i, color in enumerate(colors_test):
        row, col = divmod(i, 3)
        y1, y2 = row * (size // 2), (row + 1) * (size // 2)
        x1, x2 = col * (size // 3), (col + 1) * (size // 3)
        img_array[y1:y2, x1:x2] = color

    # Add some blur to create gradients between regions
    img_array = cv2.GaussianBlur(img_array, (31, 31), 15)
    return Image.fromarray(img_array)


# Run the demo
test_img = create_test_image(300)
result = convert_to_paint_by_number(
    test_img, n_colors=8, groq_api_key="", min_region_area=100
)

# Display results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('🎨 Paint by Number Pipeline Demo', fontsize=16, fontweight='bold')

axes[0].imshow(test_img)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(Image.open(result['outline_path']))
axes[1].set_title('Paint-by-Number Outline', fontsize=12)
axes[1].axis('off')

axes[2].imshow(Image.open(result['legend_path']))
axes[2].set_title('Color Legend', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\n🎨 Color Palette Summary:")
for i, (name, hex_code) in enumerate(zip(result['color_names'], result['hex_codes'])):
    print(f"  {i+1:2d}. {name:<30} {hex_code}")

## 🌐 Section 8: Gradio Web Application

In [ ]:
# ============================================================
# SECTION 8: GRADIO WEB APPLICATION
# ============================================================
# Defines the full Gradio UI with professional styling.
# Users can upload images, configure settings, enter a Groq
# API key for AI color naming, and download all outputs.
# ============================================================


def gradio_process(image, n_colors, groq_api_key, min_area, smoothing):
    """
    Gradio event handler that bridges the UI to the conversion pipeline.

    Accepts inputs directly from Gradio widgets, calls the main
    convert_to_paint_by_number() pipeline, and returns outputs
    formatted for Gradio's Image and File components.

    Parameters:
    -----------
    image : numpy.ndarray
        The uploaded image from Gradio's Image component (RGB).
    n_colors : int
        Number of paint colors, from the Gradio slider.
    groq_api_key : str
        Optional Groq API key entered by the user.
    min_area : int
        Minimum region area for number labeling.
    smoothing : bool
        Whether to apply morphological smoothing.

    Returns:
    --------
    tuple : (outline_img, legend_img, reference_img, color_html, files)
        - outline_img   : PIL Image — the numbered outline
        - legend_img    : PIL Image — the color legend
        - reference_img : PIL Image — the quantized reference
        - color_html    : str — HTML color palette display
        - files         : list of str — downloadable file paths
    """
    if image is None:
        return None, None, None, "<p>Please upload an image first.</p>", []

    try:
        result = convert_to_paint_by_number(
            image,
            n_colors=int(n_colors),
            groq_api_key=groq_api_key,
            min_region_area=int(min_area),
            smoothing=smoothing
        )

        # Build color palette HTML
        html = build_palette_html(result['color_names'], result['hex_codes'], result['palette'])

        outline_img   = Image.open(result['outline_path'])
        legend_img    = Image.open(result['legend_path'])
        reference_img = Image.open(result['quantized_path'])

        files = [result['outline_path'], result['legend_path'], result['quantized_path']]

        return outline_img, legend_img, reference_img, html, files

    except Exception as e:
        error_html = f'<div style="color:red;padding:1rem;">❌ Error: {str(e)}</div>'
        return None, None, None, error_html, []


def build_palette_html(color_names, hex_codes, palette):
    """
    Build an HTML color palette grid for display inside Gradio.

    Generates a responsive grid of color swatches with number badges,
    color names, hex codes, and RGB values. The grid is styled inline
    so it renders correctly within Gradio's HTML component without
    external CSS dependencies.

    Parameters:
    -----------
    color_names : list of str  — descriptive name for each color
    hex_codes   : list of str  — hex code string for each color
    palette     : numpy.ndarray (N x 3) — RGB values

    Returns:
    --------
    str : HTML string for display in gr.HTML()
    """
    swatches_html = ""
    for i, (name, hex_code, rgb) in enumerate(zip(color_names, hex_codes, palette)):
        r, g, b = int(rgb[0]), int(rgb[1]), int(rgb[2])
        # Determine if text should be dark or light based on brightness
        brightness = 0.299 * r + 0.587 * g + 0.114 * b
        text_color = "#1a1a1a" if brightness > 128 else "#f5f5f5"
        swatches_html += f"""
        <div style="display:flex;align-items:center;background:#fff;border-radius:12px;
                    padding:10px 14px;gap:12px;box-shadow:0 2px 8px rgba(0,0,0,0.08);
                    border:1px solid #eee;">
          <div style="width:50px;height:50px;border-radius:8px;background:{hex_code};
                      flex-shrink:0;border:2px solid rgba(0,0,0,0.1);position:relative;">
            <span style="position:absolute;top:2px;left:3px;font-size:11px;
                         font-weight:800;color:{text_color};">{i+1}</span>
          </div>
          <div style="flex:1;min-width:0;">
            <div style="font-weight:600;font-size:13px;color:#2d2d2d;truncate">{name}</div>
            <div style="font-size:11px;color:#888;font-family:monospace;">
              {hex_code} &nbsp;·&nbsp; rgb({r},{g},{b})
            </div>
          </div>
        </div>"""

    return f"""
    <div style="padding:8px 0;">
      <h3 style="margin:0 0 12px;font-size:15px;color:#333;font-weight:700;
                 letter-spacing:0.5px;">🎨 Suggested Paint Colors</h3>
      <div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(240px,1fr));
                  gap:8px;">{swatches_html}</div>
    </div>"""


print("✅ Gradio handler functions defined!")

In [ ]:
# ============================================================
# SECTION 8b: GRADIO UI LAYOUT & LAUNCH
# ============================================================
# Builds the complete Gradio Blocks interface with a
# professional, modern design and launches the web app.
# ============================================================

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700&family=DM+Sans:wght@400;500;600&display=swap');

body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background: #f9f7f4 !important;
}
.app-header {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    color: white;
    padding: 2rem 2.5rem;
    border-radius: 16px;
    margin-bottom: 1.5rem;
    text-align: center;
    box-shadow: 0 8px 32px rgba(0,0,0,0.2);
}
.app-header h1 {
    font-family: 'Playfair Display', serif;
    font-size: 2.2rem;
    margin: 0 0 0.5rem;
    color: #f5c842;
    letter-spacing: 1px;
}
.app-header p { color: #b0c4de; margin: 0; font-size: 0.95rem; }
.panel { background: white; border-radius: 12px; padding: 1.2rem;
         box-shadow: 0 2px 12px rgba(0,0,0,0.06); border: 1px solid #ede9e3; }
.section-label { font-weight: 600; color: #444; font-size: 0.82rem;
                 text-transform: uppercase; letter-spacing: 1px; margin-bottom: 6px; }
.run-btn { background: linear-gradient(135deg, #f5c842, #f0a500) !important;
           color: #1a1a2e !important; font-weight: 700 !important;
           border-radius: 10px !important; padding: 14px !important;
           font-size: 1rem !important; border: none !important;
           box-shadow: 0 4px 15px rgba(245,200,66,0.4) !important; }
.run-btn:hover { transform: translateY(-2px) !important; }
.tip-box { background: #f0f7ff; border-left: 4px solid #4a90d9;
           border-radius: 0 8px 8px 0; padding: 10px 14px;
           font-size: 0.82rem; color: #2c5f8a; margin-top: 8px; }
"""

with gr.Blocks(css=CSS, title="🎨 Paint by Number Converter") as demo:

    gr.HTML("""
    <div class='app-header'>
      <h1>🎨 Paint by Number Converter</h1>
      <p>Transform any photograph into a professional paint-by-number artwork — powered by K-Means color science & Groq AI</p>
    </div>
    """)

    with gr.Row(equal_height=False):

        # ─── LEFT PANEL: Controls ──────────────────────────
        with gr.Column(scale=1, min_width=300):
            gr.HTML("<div class='section-label'>📁 Upload Image</div>")
            input_image = gr.Image(
                label="", type="numpy",
                sources=["upload", "clipboard"],
                height=280, container=True
            )

            with gr.Group(elem_classes="panel"):
                gr.HTML("<div class='section-label'>⚙️ Settings</div>")

                n_colors_slider = gr.Slider(
                    minimum=4, maximum=24, value=12, step=1,
                    label="Number of Paint Colors",
                    info="More colors = more detail, harder to paint"
                )
                min_area_slider = gr.Slider(
                    minimum=50, maximum=1000, value=200, step=50,
                    label="Minimum Region Size (px)",
                    info="Ignore regions smaller than this"
                )
                smoothing_checkbox = gr.Checkbox(
                    value=True,
                    label="Apply Region Smoothing",
                    info="Reduces noise for cleaner outlines"
                )

            with gr.Group(elem_classes="panel"):
                gr.HTML("<div class='section-label'>🤖 Groq AI Color Naming (Optional)</div>")
                groq_key_input = gr.Textbox(
                    label="Groq API Key",
                    placeholder="gsk_xxxxxxxxxxxxxxxxxxxxxxxx",
                    type="password",
                    info="Get a free key at console.groq.com"
                )
                gr.HTML("""
                <div class='tip-box'>
                  💡 Without an API key, smart local color naming is still applied.
                  With Groq, you get artistic names like <em>"Tuscan Sunset"</em>.
                </div>""")

            run_btn = gr.Button(
                "🎨  Convert to Paint by Number",
                variant="primary",
                elem_classes="run-btn"
            )

        # ─── RIGHT PANEL: Outputs ──────────────────────────
        with gr.Column(scale=2):

            with gr.Tabs():
                with gr.TabItem("🖼️ Numbered Outline"):
                    outline_output = gr.Image(
                        label="Paint-by-Number Outline (Print this!)",
                        type="pil", height=420, show_download_button=True
                    )

                with gr.TabItem("🎨 Color Legend"):
                    legend_output = gr.Image(
                        label="Color Reference Legend",
                        type="pil", height=420, show_download_button=True
                    )

                with gr.TabItem("🗂️ Color Reference"):
                    reference_output = gr.Image(
                        label="Quantized Color Reference",
                        type="pil", height=420, show_download_button=True
                    )

            # Color palette HTML panel
            palette_html_output = gr.HTML(
                value="<div style='padding:2rem;text-align:center;color:#aaa;'>"
                      "Upload an image and click Convert to see your color palette.</div>"
            )

            # Download all files
            with gr.Group(elem_classes="panel"):
                gr.HTML("<div class='section-label'>⬇️ Download All Files</div>")
                files_output = gr.File(
                    label="",
                    file_count="multiple",
                    interactive=False
                )

    # ─── Examples row ─────────────────────────────────────
    gr.HTML("<hr style='border:1px solid #ede9e3;margin:1rem 0;'>")
    gr.HTML("""
    <div style='text-align:center;padding:1rem;'>
      <h3 style='font-family:Playfair Display,serif;color:#333;margin-bottom:0.5rem;'>How It Works</h3>
      <div style='display:flex;justify-content:center;gap:2rem;flex-wrap:wrap;margin-top:1rem;'>
        <div style='text-align:center;max-width:160px;'>
          <div style='font-size:2rem;'>📷</div>
          <div style='font-weight:600;color:#444;font-size:0.9rem;'>1. Upload Photo</div>
          <div style='color:#888;font-size:0.8rem;'>Any JPG, PNG, or WebP image</div>
        </div>
        <div style='font-size:1.5rem;align-self:center;color:#ccc;'>→</div>
        <div style='text-align:center;max-width:160px;'>
          <div style='font-size:2rem;'>🎯</div>
          <div style='font-weight:600;color:#444;font-size:0.9rem;'>2. K-Means Clustering</div>
          <div style='color:#888;font-size:0.8rem;'>Groups pixels into N dominant colors</div>
        </div>
        <div style='font-size:1.5rem;align-self:center;color:#ccc;'>→</div>
        <div style='text-align:center;max-width:160px;'>
          <div style='font-size:2rem;'>✏️</div>
          <div style='font-weight:600;color:#444;font-size:0.9rem;'>3. Region Outlines</div>
          <div style='color:#888;font-size:0.8rem;'>Draws clean borders around each area</div>
        </div>
        <div style='font-size:1.5rem;align-self:center;color:#ccc;'>→</div>
        <div style='text-align:center;max-width:160px;'>
          <div style='font-size:2rem;'>🔢</div>
          <div style='font-weight:600;color:#444;font-size:0.9rem;'>4. Number Labeling</div>
          <div style='color:#888;font-size:0.8rem;'>Numbers placed in each region</div>
        </div>
        <div style='font-size:1.5rem;align-self:center;color:#ccc;'>→</div>
        <div style='text-align:center;max-width:160px;'>
          <div style='font-size:2rem;'>🎨</div>
          <div style='font-weight:600;color:#444;font-size:0.9rem;'>5. Color Legend</div>
          <div style='color:#888;font-size:0.8rem;'>AI-named paints with hex codes</div>
        </div>
      </div>
    </div>""")

    # Wire up the button
    run_btn.click(
        fn=gradio_process,
        inputs=[input_image, n_colors_slider, groq_key_input,
                min_area_slider, smoothing_checkbox],
        outputs=[outline_output, legend_output, reference_output,
                 palette_html_output, files_output],
        api_name="convert"
    )

# Launch the app
demo.launch(
    share=True,
    show_error=True,
    server_name="0.0.0.0"
)

---
## 📥 Section 9: Standalone Conversion (No UI)

Use this cell to run the conversion directly on your own image without launching the Gradio UI.

In [ ]:
# ============================================================
# SECTION 9: STANDALONE CONVERSION — NO UI REQUIRED
# ============================================================
# Use this cell to convert your own image directly.
# Edit the variables below and run the cell.
# ============================================================

# ─── USER CONFIGURATION ───────────────────────────────────
INPUT_IMAGE_PATH = "your_image.jpg"   # ← Change this to your image path
OUTPUT_DIR       = "./paint_output"   # ← Where to save results
NUM_COLORS       = 12                  # ← Number of paint colors (4-24)
GROQ_API_KEY     = ""                  # ← Optional: paste your Groq key
MIN_REGION_AREA  = 200                 # ← Min pixels for a numbered region
APPLY_SMOOTHING  = True               # ← Smooth region boundaries?
# ──────────────────────────────────────────────────────────

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(INPUT_IMAGE_PATH):
    result = convert_to_paint_by_number(
        INPUT_IMAGE_PATH,
        n_colors=NUM_COLORS,
        groq_api_key=GROQ_API_KEY,
        min_region_area=MIN_REGION_AREA,
        smoothing=APPLY_SMOOTHING
    )

    # Copy outputs to user's directory
    from shutil import copy2
    copy2(result['outline_path'],   os.path.join(OUTPUT_DIR, 'outline.png'))
    copy2(result['legend_path'],    os.path.join(OUTPUT_DIR, 'legend.png'))
    copy2(result['quantized_path'], os.path.join(OUTPUT_DIR, 'reference.png'))

    print(f"✅ Files saved to: {os.path.abspath(OUTPUT_DIR)}")
    print("   outline.png   — numbered paint-by-number outline")
    print("   legend.png    — color swatch legend")
    print("   reference.png — quantized color reference")

    print("\n🎨 Your Color Palette:")
    for i, (name, hx) in enumerate(zip(result['color_names'], result['hex_codes'])):
        print(f"  {i+1:2d}. {name:<35} {hx}")
else:
    print(f"⚠️  Image not found at: {INPUT_IMAGE_PATH}")
    print("     Please update INPUT_IMAGE_PATH and re-run.")